TAU-SIGMA_R-PLOT:

In [26]:
import os
import re
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import ScatterChart, Series, Reference
from openpyxl.chart.marker import Marker
from openpyxl.chart.series import DataPoint
from openpyxl.drawing.line import LineProperties
from openpyxl.utils.dataframe import dataframe_to_rows

# 🔧 Endre denne banen til din lokale mappe
mappebane = r"\\tos-nasuni-01\GEO\Prosjekt\O10266\10266411-02\10266411-02-03 ARBEIDSOMRAADE\10266411-02 RIG\10266411-02-07 FELT- OG LABREGISTRERINGER\LAB\Treaks"

excel_filer = [f for f in os.listdir(mappebane) if f.endswith(('.xls', '.xlsx', '.xlsm'))]

wb = Workbook()
ws = wb.active
ws.title = "Samlede data"
ws_depth = wb.create_sheet("Tau_vs_Dybde")

start_col = 1
start_col_depth = 1
marker_symbols = [
    "circle",
    "triangle",
    "square",
    "diamond",
    "star",
    "plus",
    "x",
]
point_marker_symbols = [
    "circle",
    "triangle",
    "square",
    "diamond",
    "star",
    "plus",
    "x",
    "dash",
    "dot",
]

# Les og lim inn data fra hver fil
for filnavn in excel_filer:
    filsti = os.path.join(mappebane, filnavn)
    try:
        df = pd.read_excel(
            filsti,
            sheet_name='Skjær',
            usecols="V,Z",
            skiprows=19,
            nrows=1339,
            engine='openpyxl'
        )
        df = df.dropna()
        ws.cell(row=1, column=start_col, value=f"{filnavn.split('BP ')[1]}_tau")
        ws.cell(row=1, column=start_col + 1, value=f"{filnavn.split('BP ')[1]}_sigma_r")
        for i, row in enumerate(df.itertuples(index=False), start=2):
            ws.cell(row=i, column=start_col, value=row[0])
            ws.cell(row=i, column=start_col + 1, value=row[1])
        df_strain = pd.read_excel(
            filsti,
            sheet_name='1.NTNU',
            usecols="R,S",
            skiprows=15,
            nrows=9,
            engine='openpyxl'
        )
        df_strain = df_strain.dropna()
        ws.cell(row=1, column=start_col + 2, value=f"{filnavn.split('BP ')[1]}_tau_tøyning")
        ws.cell(row=1, column=start_col + 3, value=f"{filnavn.split('BP ')[1]}_sigma_r_tøyning")
        for i, row in enumerate(df_strain.itertuples(index=False), start=2):
            ws.cell(row=i, column=start_col + 2, value=row[1])
            ws.cell(row=i, column=start_col + 3, value=row[0])
        # hent dybde fra filnavn: d=7,6m
        depth_value = None
        match = re.search(r"d=([0-9]+[.,]?[0-9]*)", filnavn, re.IGNORECASE)
        if match:
            depth_str = match.group(1).replace(",", ".")
            try:
                depth_value = float(depth_str)
            except ValueError:
                depth_value = None
        if depth_value is not None and not df_strain.empty:
            ws_depth.cell(row=1, column=start_col_depth, value=f"{filnavn.split('BP ')[1]}_tau_tøyning")
            ws_depth.cell(row=1, column=start_col_depth + 1, value=f"{filnavn.split('BP ')[1]}_dybde_m")
            for i, row in enumerate(df_strain.itertuples(index=False), start=2):
                ws_depth.cell(row=i, column=start_col_depth, value=row[1])
                ws_depth.cell(row=i, column=start_col_depth + 1, value=depth_value)
            start_col_depth += 2

        start_col += 4
    except Exception as e:
        print(f"Feil ved behandling av {filnavn}: {e}")

# Lag et scatterplot i Excel (tau - sigma_r)
chart = ScatterChart()
chart.scatterStyle = "lineMarker"  # tillater linjer og markører; vi slår av/på per serie
chart.title = "Samleplott av skjærdata Treaks"
chart.x_axis.title = "Maks. skjærspenning, τ_max (kPa)"
chart.y_axis.title = "Eff. radiell spenning, σ'r (kPa)"
chart.x_axis.scaling.orientation = "maxMin"

for col in range(1, start_col, 4):
    xvalues = Reference(ws, min_col=col, min_row=2, max_row=ws.max_row)
    yvalues = Reference(ws, min_col=col+1, min_row=2, max_row=ws.max_row)
    series = Series(yvalues, xvalues, title=ws.cell(row=1, column=col).value)
    series.marker = Marker(symbol="none")  # hovedserie: ingen markører, linje beholdes
    chart.series.append(series)

    extra_x = Reference(ws, min_col=col+2, min_row=2, max_row=10)
    extra_y = Reference(ws, min_col=col+3, min_row=2, max_row=10)
    extra_series = Series(extra_y, extra_x, title=f"Tøyning 0,5-10% {ws.cell(row=1, column=col).value}")
    extra_series.marker = None  # vi setter per punkt
    extra_series.graphicalProperties.line = LineProperties(noFill=True)

    rows_count = 0
    for r in range(2, ws.max_row + 1):
        if ws.cell(row=r, column=col+2).value is None or ws.cell(row=r, column=col+3).value is None:
            break
        rows_count += 1

    for idx in range(rows_count):
        dp = DataPoint(idx=idx)
        dp.marker = Marker(symbol=point_marker_symbols[idx % len(point_marker_symbols)], size=10)
        extra_series.dPt.append(dp)

    chart.series.append(extra_series)

ws.add_chart(chart, "D10")

# Lag plott tau vs dybde
depth_chart = ScatterChart()
depth_chart.scatterStyle = "marker"
depth_chart.title = "Tau vs dybde, tøyning 0,5-10%"
depth_chart.x_axis.title = "τ (kPa)"
depth_chart.y_axis.title = "Dybde (m)"
depth_chart.y_axis.scaling.orientation = "maxMin"
for col in range(1, start_col_depth, 2):
    xvalues = Reference(ws_depth, min_col=col, min_row=2, max_row=ws_depth.max_row)
    yvalues = Reference(ws_depth, min_col=col+1, min_row=2, max_row=ws_depth.max_row)
    series = Series(yvalues, xvalues, title=ws_depth.cell(row=1, column=col).value)
    series.marker = Marker(symbol="circle", size=8)
    depth_chart.series.append(series)

ws_depth.add_chart(depth_chart, "E2")

wb.save(mappebane + "\\samledata_tau_sigma_r.xlsx")
print("Excel-fil med samlede data og plott er lagret som 'samledata_tau_sigma_r.xlsx'")


Feil ved behandling av ~$10266411-02 BP 15 d=10,6m CIUp.xlsm: File is not a zip file
Feil ved behandling av ~$10266411-02 BP 8 d=6,8m CIUp.xlsm: File is not a zip file
Feil ved behandling av ~$10266411-02 BP 18 d=3,55m CIUp.xlsm: File is not a zip file
Feil ved behandling av ~$10266411-02 BP 10 d=6,55m CAUa.xlsm: File is not a zip file
Feil ved behandling av Lokal-samledata_p-q.xlsx: Worksheet named 'Skjær' not found
Feil ved behandling av ~$Lokal-samledata_p-q.xlsx: File is not a zip file
Feil ved behandling av Tøyning_samledata_tau_sigma_r.xlsx: Worksheet named 'Skjær' not found
Feil ved behandling av samledata_tau_sigma_r.xlsx: Worksheet named 'Skjær' not found
Feil ved behandling av samledata_tau_sigma_r_bytte.xlsx: Worksheet named 'Skjær' not found
Excel-fil med samlede data og plott er lagret som 'samledata_tau_sigma_r.xlsx'


p-q-PLOT:

In [ ]:
import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import ScatterChart, Series, Reference
from openpyxl.chart.marker import Marker
from openpyxl.chart.series import DataPoint
from openpyxl.drawing.line import LineProperties
from openpyxl.utils.dataframe import dataframe_to_rows

mappebane = r"\\tos-nasuni-01\GEO\Prosjekt\O10266\10266411-02\10266411-02-03 ARBEIDSOMRAADE\10266411-02 RIG\10266411-02-07 FELT- OG LABREGISTRERINGER\LAB\Treaks"

excel_filer = [f for f in os.listdir(mappebane) if f.endswith(('.xls', '.xlsx', '.xlsm'))]

wb = Workbook()
ws = wb.active
ws.title = "Samlede data"

start_col = 1
point_marker_symbols = [
    "circle",
    "triangle",
    "square",
    "diamond",
    "star",
    "plus",
    "x",
    "dash",
    "dot",
]

# Les og lim inn data fra hver fil
for filnavn in excel_filer:
    filsti = os.path.join(mappebane, filnavn)
    try:
        df = pd.read_excel(
            filsti,
            sheet_name='Skjær',
            usecols="U,AC",
            skiprows=19,
            nrows=1339,
            engine='openpyxl'
        )
        df = df.dropna()
        ws.cell(row=1, column=start_col, value=f"{filnavn.split('BP ')[1]}_q")
        ws.cell(row=1, column=start_col + 1, value=f"{filnavn.split('BP ')[1]}_p")
        for i, row in enumerate(df.itertuples(index=False), start=2):
            ws.cell(row=i, column=start_col, value=row[0])
            ws.cell(row=i, column=start_col + 1, value=row[1])
        df_strain = pd.read_excel(
            filsti,
            sheet_name='1.NTNU',
            usecols="V,U",
            skiprows=15,
            nrows=9,
            engine='openpyxl'
        )
        df_strain = df_strain.dropna()
        ws.cell(row=1, column=start_col + 2, value=f"{filnavn.split('BP ')[1]}_q_tøyning")
        ws.cell(row=1, column=start_col + 3, value=f"{filnavn.split('BP ')[1]}_p_tøyning")
        for i, row in enumerate(df_strain.itertuples(index=False), start=2):
            ws.cell(row=i, column=start_col + 2, value=row[1])
            ws.cell(row=i, column=start_col + 3, value=row[0])

        start_col += 4
    except Exception as e:
        print(f"Feil ved behandling av {filnavn}: {e}")

# Lag et scatterplot i Excel
chart = ScatterChart()
chart.scatterStyle = "line"
chart.title = "Samleplott av skjærdata Treaks"
chart.x_axis.title = "Deviatorspenning, q (kPa)"
chart.y_axis.title = "Effektiv middelspenning, p' (kPa)"
chart.x_axis.scaling.orientation = "maxMin"

for col in range(1, start_col, 4):
    xvalues = Reference(ws, min_col=col, min_row=2, max_row=ws.max_row)
    yvalues = Reference(ws, min_col=col+1, min_row=2, max_row=ws.max_row)
    series = Series(yvalues, xvalues, title=ws.cell(row=1, column=col).value)
    series.marker = None
    chart.series.append(series)

    extra_x = Reference(ws, min_col=col+2, min_row=2, max_row=10)
    extra_y = Reference(ws, min_col=col+3, min_row=2, max_row=10)
    extra_series = Series(extra_y, extra_x, title=f"Tøyning 0,5-10% {ws.cell(row=1, column=col).value}")
    extra_series.marker = None
    lp = LineProperties(noFill=True)
    lp.width = 0
    extra_series.graphicalProperties.line = lp

    rows_count = 0
    for r in range(2, ws.max_row + 1):
        if ws.cell(row=r, column=col+2).value is None or ws.cell(row=r, column=col+3).value is None:
            break
        rows_count += 1

    for idx in range(rows_count):
        dp = DataPoint(idx=idx)
        dp.marker = Marker(symbol=point_marker_symbols[idx % len(point_marker_symbols)], size=10)
        extra_series.dPt.append(dp)

    chart.series.append(extra_series)

ws.add_chart(chart, "D10")

wb.save(mappebane + "\\10266411-samledata_p-q.xlsx")
print("Excel-fil med samlede data og plott er lagret som '10266411-samledata_p-q.xlsx'")


Feil ved behandling av ~$10266411-02 BP 15 d=10,6m CIUp.xlsm: File is not a zip file
Feil ved behandling av ~$10266411-02 BP 8 d=6,8m CIUp.xlsm: File is not a zip file


KeyboardInterrupt: 